# 02 Grating Tuning and PO Estimation

This notebook explains the most important sanity check in the project. A full-field grating translated by gaze does **not** rotate in visual space. It changes the phase seen by each RF:

```text
Δφ = 2π f (Δa cosθ + Δe sinθ)
```

Therefore, with perfect mapping and no noise, a phase-invariant energy model or a densely phase-averaged simple-cell model should show little deterministic ΔPO. Apparent ΔPO for gratings is expected mainly when finite phase sampling, rectification, mapping error, or noise changes the estimated tuning curve.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.utils import load_config, rng_from_config
from src.sampling import sample_population
from src.stimuli import make_grating_stimuli
from src.responses import grating_response_mean
from src.tuning import estimate_orientation_tuning
from src.simulation import perfect_mapping_config, run_grating_shift_grid
from src.plotting import set_plot_style, plot_grating_gaze_rf_scheme

set_plot_style()
fig_dir = ROOT / 'results/figures'
fig_dir.mkdir(parents=True, exist_ok=True)

cfg = load_config(ROOT / 'configs/default.yaml', debug=True)
rng = rng_from_config(cfg)
pop = sample_population(cfg, rng)
stim = make_grating_stimuli(cfg)
print('orientations:', stim['orientations_deg'])
print('spatial frequencies:', np.round(stim['spatial_frequencies_cpd'], 4))
print('phases (deg):', np.round(np.rad2deg(stim['phases_rad']), 1))

## Grating-Gaze-RF Scheme

This schematic is the conceptual bridge for the rest of the notebook. The RF stays fixed in visual coordinates. A gaze/FOV shift translates the grating over that RF. The grating orientation is unchanged, but the phase at the RF changes in an orientation-dependent way. Sparse phase-sensitive responses can therefore reshape the estimated tuning curve and move inferred PO.

In [ ]:
scheme_path = fig_dir / 'notebook02_grating_gaze_rf_scheme.png'
plot_grating_gaze_rf_scheme(scheme_path)
from IPython.display import Image, display
display(Image(filename=str(scheme_path)))

## Phase Advance From a Gaze/FOV Shift

The phase advance depends on grating orientation and SF. This is the only deterministic effect of gaze for an ideal infinite grating under perfect mapping. The plot below shows that the same 5 degree azimuth drift creates different phase advances for different grating orientations.

In [ ]:
orientations = np.linspace(0, 170, 18)
drift = (5.0, 0.0)
fig, ax = plt.subplots(figsize=(7, 4))
for sf in [0.02, 0.06, 0.12, 0.20]:
    phase_cycles = sf * (drift[0] * np.cos(np.deg2rad(orientations)) + drift[1] * np.sin(np.deg2rad(orientations)))
    ax.plot(orientations, phase_cycles, marker='o', label=f'{sf:.2f} cpd')
ax.axhline(0, color='0.25', lw=1)
ax.set_xlabel('Grating orientation θ (deg)')
ax.set_ylabel('Phase advance Δφ / 2π (cycles)')
ax.set_title('A gaze shift changes grating phase, not grating orientation')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(fig_dir / 'notebook02_phase_advance_by_orientation.png', bbox_inches='tight')

## Example Tuning Curves

The simple-cell model is phase-sensitive after rectification. The energy model uses a quadrature pair and removes phase sensitivity. The two rows below are intentionally compared at the same gaze shift.

In [ ]:
gaze = (5.0, 5.0)
simple_base, _ = grating_response_mean(pop, stim, cfg, rng, gaze_shift=(0, 0), response_model='simple')
simple_shift, _ = grating_response_mean(pop, stim, cfg, rng, gaze_shift=gaze, response_model='simple')
energy_base, _ = grating_response_mean(pop, stim, cfg, rng, gaze_shift=(0, 0), response_model='energy')
energy_shift, _ = grating_response_mean(pop, stim, cfg, rng, gaze_shift=gaze, response_model='energy')

simple_base_t = estimate_orientation_tuning(simple_base, stim['orientations_deg'], stim['spatial_frequencies_cpd'])
simple_shift_t = estimate_orientation_tuning(simple_shift, stim['orientations_deg'], stim['spatial_frequencies_cpd'])
energy_base_t = estimate_orientation_tuning(energy_base, stim['orientations_deg'], stim['spatial_frequencies_cpd'])
energy_shift_t = estimate_orientation_tuning(energy_shift, stim['orientations_deg'], stim['spatial_frequencies_cpd'])

examples = [0, len(pop)//2, len(pop)-1]
fig, axes = plt.subplots(2, len(examples), figsize=(12, 6), sharex=True)
for col, neuron in enumerate(examples):
    axes[0, col].plot(stim['orientations_deg'], simple_base_t['tuning_curve'][neuron], 'o-', label='0 deg')
    axes[0, col].plot(stim['orientations_deg'], simple_shift_t['tuning_curve'][neuron], 's-', label='5,5 deg')
    axes[0, col].set_title(f'simple cell {neuron}')
    axes[1, col].plot(stim['orientations_deg'], energy_base_t['tuning_curve'][neuron], 'o-', label='0 deg')
    axes[1, col].plot(stim['orientations_deg'], energy_shift_t['tuning_curve'][neuron], 's-', label='5,5 deg')
    axes[1, col].set_title(f'energy cell {neuron}')
    axes[1, col].set_xlabel('Orientation (deg)')
axes[0, 0].set_ylabel('Response'); axes[1, 0].set_ylabel('Response')
axes[0, 0].legend(frameon=False); axes[1, 0].legend(frameon=False)
plt.tight_layout()
plt.savefig(fig_dir / 'notebook02_example_tuning_simple_vs_energy.png', bbox_inches='tight')

## Phase-Sampling Sanity Check

If the model is behaving as intended, `|ΔPO|` should be largest for phase-sensitive simple-cell estimates with few phases, shrink with dense phase averaging, and be essentially zero for the energy model. This is the strongest sanity check for the full-field grating interpretation.

In [ ]:
drifts = np.array([0.0, 1.0, 2.0, 5.0, 10.0])
shifts = pd.DataFrame({'gaze_az_deg': drifts, 'gaze_el_deg': drifts})
conditions = [
    ('simple: 1 phase', 'simple', [0.0]),
    ('simple: 2 phases', 'simple', [0.0, 90.0]),
    ('simple: 4 phases', 'simple', [0.0, 90.0, 180.0, 270.0]),
    ('simple: 24 phases', 'simple', list(np.linspace(0, 360, 24, endpoint=False))),
    ('energy model', 'energy', [0.0]),
]
rows = []
for label, model, phases in conditions:
    local_cfg = deepcopy(cfg)
    local_cfg['stimuli']['gratings']['phases_deg'] = phases
    result = run_grating_shift_grid(pop, local_cfg, rng, mapping_config=perfect_mapping_config(), noise_config={'model': 'none', 'repeats': 1}, response_model=model, shifts=shifts)['summary']
    result['condition'] = label
    result['drift_deg'] = result['gaze_az_deg']
    rows.append(result)
phase_sanity = pd.concat(rows, ignore_index=True)
display(phase_sanity.pivot(index='drift_deg', columns='condition', values='median_abs_delta_po_deg').round(3))

plt.figure(figsize=(7.5, 4.8))
sns.lineplot(data=phase_sanity, x='drift_deg', y='median_abs_delta_po_deg', hue='condition', marker='o')
plt.xlabel('Diagonal gaze/FOV drift (deg)')
plt.ylabel('Median |ΔPO| (deg)')
plt.title('Full-field grating sanity check: phase sensitivity creates apparent ΔPO')
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(fig_dir / 'notebook02_phase_sampling_sanity.png', bbox_inches='tight')

## Orientation-by-SF Response Surface

The estimated PO is extracted from orientation tuning at the best SF. Looking at the full orientation-by-SF surface helps distinguish a true peak shift from general flattening or low-SNR instability.

In [ ]:
neuron = examples[0]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for ax, matrix, title in zip(axes, [simple_base[neuron], simple_shift[neuron]], ['baseline', 'shifted gaze']):
    im = ax.imshow(matrix, aspect='auto', origin='lower', cmap='mako', extent=[stim['spatial_frequencies_cpd'][0], stim['spatial_frequencies_cpd'][-1], stim['orientations_deg'][0], stim['orientations_deg'][-1]])
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel('SF (cpd)')
axes[0].set_ylabel('Orientation (deg)')
fig.colorbar(im, ax=axes, label='Response')
plt.savefig(fig_dir / 'notebook02_orientation_sf_surface.png', bbox_inches='tight')

## Takeaway

For full-field gratings, gaze-induced ΔPO is not caused by the grating rotating. It is an apparent tuning-estimation effect caused by how phase-dependent responses are sampled and summarized. This is why the project always compares simple-cell and energy-model responses and separates deterministic gaze, mapping error, and noise.